# Phase 02 — Data Audit (future task)

**STOP:** Phase 01 Business Understanding must be reviewed before using this notebook.

Data source: [Olist dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).

The dataset is **not** stored in this repository. You will supply it from your own cloud storage during Phase 02.

## Task 1 — Define the Grain and Keys

### 1. Orders Table
- **Expected grain:** One row per order.
- **Expected primary key:** `order_id`.
- **Expected uniqueness:** Each `order_id` should appear only once.
- **Expected relationships:** An order may be linked to one customer and may contain multiple order items.

### 2. Customers Table
- **Expected grain:** One row per customer record.
- **Expected primary key:** `customer_id`.
- **Expected uniqueness:** Each `customer_id` is expected to appear only once.
- **Expected relationships:** A customer may place multiple orders. The `customer_id` in the orders table is expected to link to the customers table.

### 3. Order Items Table
- **Expected grain:** One row per item within an order.
- **Expected composite key:** (`order_id`, `order_item_id`).
- **Expected uniqueness:** Each combination of `order_id` and `order_item_id` should appear only once.
- **Expected relationships:** An order may contain multiple items, and each item record is expected to link to an order through `order_id`. Products and sellers may also be linked through their respective IDs.

### Audit Goal
After loading the data, I will check whether the actual grain, key uniqueness, and relationships match these expectations. These are initial assumptions and must be verified using the data.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")

print("Data directory:", DATA_DIR.resolve())
print("Directory exists:", DATA_DIR.exists())

Data directory: D:\olist-retail-intelligence\data\raw
Directory exists: True


In [6]:
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")

In [7]:
print("Orders shape:", orders.shape)
print("Customers shape:", customers.shape)
print("Order Items shape:", order_items.shape)

Orders shape: (99441, 8)
Customers shape: (99441, 5)
Order Items shape: (112650, 7)


In [8]:
print("ORDERS")
print(orders.dtypes)

print("\nCUSTOMERS")
print(customers.dtypes)

print("\nORDER ITEMS")
print(order_items.dtypes)

ORDERS
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

CUSTOMERS
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

ORDER ITEMS
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object


In [16]:
for name,df in{"ordes":orders,"customers":customers,"order_items":order_items}.items():
    print(f"\n{name.upper()}")
    print("Missing values:")
    print(df.isnull().sum())


ORDES
Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

CUSTOMERS
Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

ORDER_ITEMS
Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64


In [17]:
orders.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ]
].agg(lambda col: col.isna().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [18]:
for name,df in{"ordes":orders,"customers":customers,"order_items":order_items}.items():
     print(f"{name}: {df.duplicated().sum()} exact duplicate rows")

ordes: 0 exact duplicate rows
customers: 0 exact duplicate rows
order_items: 0 exact duplicate rows


In [19]:
print("Orders — duplicate order_id:",
      orders["order_id"].duplicated().sum())

print("Customers — duplicate customer_id:",
      customers["customer_id"].duplicated().sum())

print("Order Items — duplicate composite key:",
      order_items.duplicated(
          subset=["order_id", "order_item_id"]
      ).sum())

Orders — duplicate order_id: 0
Customers — duplicate customer_id: 0
Order Items — duplicate composite key: 0


In [23]:
print(
    "Orders with unmatched customer_id:",
    (~orders["customer_id"].isin(customers["customer_id"])).sum()
)

print(
    "Order items with unmatched order_id:",
    (~order_items["order_id"].isin(orders["order_id"])).sum()
)

Orders with unmatched customer_id: 0
Order items with unmatched order_id: 0


## Task 2 — Audit data shape, column types, nulls and duplicates

### 1. Data Shape
- `orders`: 99,441 rows and 8 columns.
- `customers`: 99,441 rows and 5 columns.
- `order_items`: 112,650 rows and 7 columns.
- The larger number of rows in `order_items` is consistent with an order containing multiple items.

### 2. Column Types
- ID columns are stored as `object`, which is appropriate for identifiers.
- Date-related columns are stored as `object` and should be reviewed before any conversion.
- `customer_zip_code_prefix` is stored as `int64`; its format should be reviewed because it represents a postal code prefix.
- `order_item_id` is stored as `int64`, while `price` and `freight_value` are stored as `float64`.

### 3. Missing Values
- `orders` contains missing values in `order_approved_at` (160), `order_delivered_carrier_date` (1,783), and `order_delivered_customer_date` (2,965).
- `customers` and `order_items` contain no missing values in the inspected columns.
- Missing delivery dates vary by `order_status`. Some missing values may be expected for orders that have not reached the delivery stage, while missing dates in `delivered` orders require further investigation.

### 4. Duplicates and Key Checks
- No exact duplicate rows were found in the three inspected tables.
- No duplicate `order_id` values were found in `orders`.
- No duplicate `customer_id` values were found in `customers`.
- No duplicate (`order_id`, `order_item_id`) combinations were found in `order_items`.

### 5. Relationship Checks
- All `customer_id` values in `orders` matched records in `customers`.
- All `order_id` values in `order_items` matched records in `orders`.
- These checks confirm the tested relationships, but do not validate every possible relationship in the full dataset.

### Audit Decision
- No rows or missing values were removed or modified.
- Further checks and investigation are needed before making cleaning decisions.

### Task 3 — The Danger of Joining One Order to Multiple Items

#### Observations
- The `orders` table contains 99,441 rows, while `order_items` contains 112,650 rows.
- An inner join on `order_id` produces 112,650 rows and 98,666 unique orders.
- The row count increases because one order can contain multiple items, causing order-level information to appear in multiple rows.
- The inner join excludes 775 orders that have no matching records in `order_items`.
- Of these 775 orders, 603 have status `unavailable`, 164 are `canceled`, 5 are `created`, 2 are `invoiced`, and 1 is `shipped`.

#### Why This Matters
- Counting rows after the join does not give the correct number of unique orders.
- Order-level values repeated across item rows may be double-counted when aggregated.
- An inner join can exclude orders without matching item records, so the resulting dataset may not represent every order in the original table.

#### Key Takeaway
Before calculating business metrics after a join, identify the grain of each table, choose the appropriate aggregation level, and check whether the join excludes records.

In [24]:
orders_items = orders.merge(
    order_items,
    on="order_id",
    how="inner"
)

print("Orders rows:", len(orders))
print("Order items rows:", len(order_items))
print("Rows after merge:", len(orders_items))
print("Unique orders after merge:", orders_items["order_id"].nunique())# Add checks if useful.


Orders rows: 99441
Order items rows: 112650
Rows after merge: 112650
Unique orders after merge: 98666


In [25]:
orders_without_items = orders.loc[
    ~orders["order_id"].isin(order_items["order_id"])
]

print("Orders without items:", len(orders_without_items))

print(
    orders_without_items["order_status"].value_counts()
)

Orders without items: 775
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


## Review notes

### 1. Key Findings
- The `orders` table contains 99,441 rows, `customers` contains 99,441 rows, and `order_items` contains 112,650 rows.
- No exact duplicate rows were found in the three inspected tables.
- The expected keys were unique: `order_id` in `orders`, `customer_id` in `customers`, and (`order_id`, `order_item_id`) in `order_items`.
- All `customer_id` values in `orders` matched records in `customers`, and all `order_id` values in `order_items` matched records in `orders`.
- Missing values were found in three date columns in `orders`. Their relationship with order status requires further investigation.
- A total of 775 orders had no matching records in `order_items`. Most were `unavailable` or `canceled`, while a small number had other statuses.
- Joining orders with order items increased the row count because one order can contain multiple items. The inner join also excluded orders without matching item records.

### 2. Open Questions
- Why do some orders with status `delivered` have missing delivery dates?
- Why do 2 `invoiced` orders and 1 `shipped` order have no matching item records?
- Are the missing dates and orders without items expected outcomes of the business process, or do they indicate data quality issues?
- Should date columns and postal code prefixes be converted or formatted differently during later preparation?

### 3. Limits of the Audit
- This audit focused on the `orders`, `customers`, and `order_items` tables.
- The checks covered data shape, column types, missing values, exact duplicates, key uniqueness, and selected relationships.
- The audit did not verify every relationship across all nine source tables.
- The causes of missing values and orders without items have not been confirmed.
- No rows were removed, and no missing values were imputed or modified.

### Next Steps
- Investigate the open questions before making cleaning decisions.
- Continue auditing the remaining tables and validate additional relationships.
- Apply cleaning and type conversions only after understanding the business meaning of the data.